# Лабораторная работа 1. Линейная регрессия и факторный анализ

Датасет: **Medical Cost Personal Dataset** (`insurance.csv`).

Цель работы - предсказать медицинские расходы пациента (`charges`) и сравнить линейную, гребневую и лассо-регрессию до и после перехода к главным компонентам.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'output'
DATA_PATH = DATA_DIR / 'insurance.csv'
DATA_URL = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'

RANDOM_STATE = 42
TEST_SIZE = 0.2
N_SPLITS = 5
VARIANCE_THRESHOLD = 0.95

FEATURE_NAMES_RU = {
    'age': 'Возраст',
    'sex_male': 'Пол: мужчина',
    'bmi': 'Индекс массы тела',
    'children': 'Количество детей',
    'smoker_yes': 'Курит',
    'region_northwest': 'Регион: northwest',
    'region_southeast': 'Регион: southeast',
    'region_southwest': 'Регион: southwest',
    'charges': 'Медицинские расходы',
}

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')

## 1. Загрузка датасета

Датасет скачивается в папку `data`. Если файл уже есть, повторная загрузка не выполняется.

In [ ]:
if not DATA_PATH.exists() or DATA_PATH.stat().st_size == 0:
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

raw = pd.read_csv(DATA_PATH)
print(f'Размер датасета: {raw.shape[0]} строк, {raw.shape[1]} столбцов')
display(raw.head())

## 2. Первичный анализ

Проверим типы данных, пропуски и основные описательные статистики. В датасете есть числовые признаки (`age`, `bmi`, `children`) и категориальные признаки (`sex`, `smoker`, `region`). Целевая переменная - `charges`.

In [ ]:
df = raw.copy()

print('Типы данных:')
display(df.dtypes)

print('Количество пропусков:')
display(df.isna().sum())

display(df.describe(include='all'))

## 3. Визуализация распределений

Построим распределения признаков и целевой переменной. Отдельно посмотрим связь расходов с курением: это один из самых сильных факторов в датасете.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(17, 8))
sns.histplot(df['age'], bins=25, ax=axes[0, 0], color='#4C78A8')
axes[0, 0].set_title('Возраст')
sns.histplot(df['bmi'], bins=25, ax=axes[0, 1], color='#4C78A8')
axes[0, 1].set_title('BMI')
sns.histplot(df['children'], bins=6, ax=axes[0, 2], color='#4C78A8')
axes[0, 2].set_title('Количество детей')
sns.histplot(df['charges'], bins=35, ax=axes[0, 3], color='#4C78A8')
axes[0, 3].set_title('Медицинские расходы')
sns.countplot(data=df, x='sex', ax=axes[1, 0], color='#F58518')
axes[1, 0].set_title('Пол')
sns.countplot(data=df, x='smoker', ax=axes[1, 1], color='#F58518')
axes[1, 1].set_title('Курение')
sns.countplot(data=df, x='region', ax=axes[1, 2], color='#F58518')
axes[1, 2].set_title('Регион')
axes[1, 2].tick_params(axis='x', rotation=20)
sns.boxplot(data=df, x='smoker', y='charges', ax=axes[1, 3])
axes[1, 3].set_title('Расходы и курение')
for ax in axes.ravel():
    ax.set_xlabel('')
    ax.set_ylabel('')
plt.suptitle('Распределения признаков и целевой переменной', y=1.02, fontsize=16)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_distributions.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='bmi', y='charges', hue='smoker', alpha=0.75)
plt.title('Связь медицинских расходов с BMI и курением')
plt.xlabel('BMI')
plt.ylabel('Медицинские расходы')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_target_vs_bmi_smoker.png', dpi=180, bbox_inches='tight')
plt.show()

## 4. Подготовка данных

Разделим признаки и целевую переменную. Числовые признаки стандартизируем, категориальные признаки закодируем методом one-hot с удалением первой категории, чтобы не получить фиктивную ловушку.

In [ ]:
X = df.drop(columns='charges')
y = df['charges']

numeric_features = ['age', 'bmi', 'children']
categorical_features = ['sex', 'smoker', 'region']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_features),
])

encoded = preprocessor.fit_transform(X)
feature_names = [name.replace('num__', '').replace('cat__', '') for name in preprocessor.get_feature_names_out()]
encoded_df = pd.DataFrame(encoded, columns=feature_names)

print('Признаки после кодирования:')
display(encoded_df.head())

## 5. Корреляционная матрица и VIF

Корреляционная матрица показывает парные связи признаков с целевой переменной. VIF оценивает, насколько каждый признак линейно объясняется остальными признаками. Обычно `VIF > 5` считают заметной мультиколлинеарностью, `VIF > 10` - сильной.

In [ ]:
corr = encoded_df.copy()
corr['charges'] = y.values
corr_matrix = corr.corr(numeric_only=True)
display(corr_matrix)

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_matrix.rename(index=FEATURE_NAMES_RU, columns=FEATURE_NAMES_RU),
    cmap='coolwarm',
    center=0,
    annot=True,
    fmt='.2f',
)
plt.title('Корреляционная матрица после кодирования категорий')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_correlation.png', dpi=180, bbox_inches='tight')
plt.show()

vif = pd.DataFrame({
    'feature': feature_names,
    'feature_ru': [FEATURE_NAMES_RU.get(feature, feature) for feature in feature_names],
    'VIF': [variance_inflation_factor(encoded_df.values, i) for i in range(encoded_df.shape[1])],
}).sort_values('VIF', ascending=False)
display(vif)

plt.figure(figsize=(9, 6))
sns.barplot(data=vif, x='VIF', y='feature_ru', color='#F58518')
plt.axvline(5, color='#E45756', linestyle='--', label='VIF = 5')
plt.axvline(10, color='#B279A2', linestyle='--', label='VIF = 10')
plt.title('Коэффициенты инфляции дисперсии до PCA')
plt.xlabel('VIF')
plt.ylabel('')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_vif.png', dpi=180, bbox_inches='tight')
plt.show()

## 6. Модели на исходных признаках

Обучим три модели: обычную линейную регрессию, Ridge и Lasso. Для Ridge и Lasso коэффициент регуляризации `alpha` подбирается по сетке с 5-fold кросс-валидацией. Метрики: RMSE, R² и MAPE.

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def evaluate(model, X_train, X_test, y_train, y_test, model_title, feature_set):
    scoring = {
        'rmse': 'neg_root_mean_squared_error',
        'r2': 'r2',
        'mape': 'neg_mean_absolute_percentage_error',
    }
    cv_scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    fitted = model.fit(X_train, y_train)
    best_model = fitted.best_estimator_ if isinstance(fitted, GridSearchCV) else fitted
    pred = best_model.predict(X_test)
    alpha = fitted.best_params_['model__alpha'] if isinstance(fitted, GridSearchCV) else None
    return best_model, {
        'feature_set': feature_set,
        'model': model_title,
        'alpha': alpha,
        'CV_RMSE': -cv_scores['test_rmse'].mean(),
        'CV_R2': cv_scores['test_r2'].mean(),
        'CV_MAPE': -cv_scores['test_mape'].mean() * 100,
        'RMSE': rmse(y_test, pred),
        'R2': r2_score(y_test, pred),
        'MAPE': mean_absolute_percentage_error(y_test, pred) * 100,
    }


def make_models(n_components=None):
    alphas = np.logspace(-3, 3, 50)
    steps = [('preprocessor', preprocessor)]
    if n_components is not None:
        steps.append(('pca', PCA(n_components=n_components)))
    return {
        'Линейная': Pipeline([*steps, ('model', LinearRegression())]),
        'Гребневая': GridSearchCV(
            Pipeline([*steps, ('model', Ridge())]),
            {'model__alpha': alphas},
            cv=N_SPLITS,
            scoring='neg_root_mean_squared_error',
        ),
        'Лассо': GridSearchCV(
            Pipeline([*steps, ('model', Lasso(max_iter=50000, random_state=RANDOM_STATE))]),
            {'model__alpha': alphas},
            cv=N_SPLITS,
            scoring='neg_root_mean_squared_error',
        ),
    }

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)
cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

rows = []
models_original = {}
for title, model in make_models().items():
    fitted, row = evaluate(model, X_train, X_test, y_train, y_test, title, 'Исходные признаки')
    models_original[title] = fitted
    rows.append(row)

metrics_original = pd.DataFrame(rows)
display(metrics_original)

In [ ]:
coef_rows = []
for model_title, pipeline in models_original.items():
    model = pipeline.named_steps['model']
    for feature, coef in zip(feature_names, model.coef_):
        coef_rows.append({
            'feature': feature,
            'feature_ru': FEATURE_NAMES_RU.get(feature, feature),
            'model': model_title,
            'coefficient': coef,
        })

coefficients_original = pd.DataFrame(coef_rows)
display(coefficients_original.pivot(index='feature_ru', columns='model', values='coefficient'))

## 7. PCA: устранение мультиколлинеарности

Перед PCA используются те же преобразования признаков: стандартизация числовых столбцов и one-hot кодирование категорий. Число компонент выбирается так, чтобы сохранить не менее 95% дисперсии.

In [ ]:
preprocessor_for_pca = preprocessor.fit(X_train)
X_train_encoded = preprocessor_for_pca.transform(X_train)

pca_full = PCA().fit(X_train_encoded)
cumulative = np.cumsum(pca_full.explained_variance_ratio_)
n_components_95 = int(np.argmax(cumulative >= VARIANCE_THRESHOLD) + 1)
print(f'Компонент для сохранения {VARIANCE_THRESHOLD:.0%} дисперсии:', n_components_95)

pca_variance = pd.DataFrame({
    'component': [f'PC{i}' for i in range(1, len(cumulative) + 1)],
    'explained_variance_ratio': pca_full.explained_variance_ratio_,
    'cumulative_variance': cumulative,
})
display(pca_variance)

component_numbers = range(1, len(cumulative) + 1)
plt.figure(figsize=(8, 5))
plt.plot(component_numbers, pca_full.explained_variance_ratio_, marker='o', label='Доля дисперсии')
plt.plot(component_numbers, cumulative, marker='s', label='Накопленная доля')
plt.axhline(VARIANCE_THRESHOLD, color='red', linestyle='--', label='95%')
plt.xticks(list(component_numbers))
plt.xlabel('Номер главной компоненты')
plt.ylabel('Доля дисперсии')
plt.title('График каменистой осыпи')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '05_scree.png', dpi=180, bbox_inches='tight')
plt.show()

pca_train = PCA(n_components=n_components_95).fit_transform(X_train_encoded)
vif_after_pca = pd.DataFrame({
    'component': [f'PC{i}' for i in range(1, n_components_95 + 1)],
    'VIF': [variance_inflation_factor(pca_train, i) for i in range(n_components_95)],
})
display(vif_after_pca)

## 8. Модели на главных компонентах

Повторим обучение тех же трех моделей, но теперь на вход подаются главные компоненты. Затем сравним метрики с моделями на исходных признаках.

In [ ]:
rows_pca = []
models_pca = {}
for title, model in make_models(n_components_95).items():
    fitted, row = evaluate(model, X_train, X_test, y_train, y_test, title, 'Главные компоненты')
    models_pca[title] = fitted
    rows_pca.append(row)

metrics = pd.concat([metrics_original, pd.DataFrame(rows_pca)], ignore_index=True)
display(metrics)

plot_metrics = metrics.copy()
plot_metrics['variant'] = plot_metrics['feature_set'] + ' / ' + plot_metrics['model']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, metric, title in zip(axes, ['RMSE', 'R2', 'MAPE'], ['RMSE', 'R²', 'MAPE, %']):
    sns.barplot(data=plot_metrics, x=metric, y='variant', ax=ax, hue='feature_set', dodge=False)
    ax.set_title(title)
    ax.set_ylabel('')
    if ax.legend_:
        ax.legend_.remove()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '06_metrics.png', dpi=180, bbox_inches='tight')
plt.show()

pred_original = models_original['Линейная'].predict(X_test)
pred_pca = models_pca['Линейная'].predict(X_test)
plt.figure(figsize=(7, 7))
plt.scatter(y_test, pred_original, alpha=0.7, label='Исходные признаки')
plt.scatter(y_test, pred_pca, alpha=0.7, label='PCA')
min_value = min(y_test.min(), pred_original.min(), pred_pca.min())
max_value = max(y_test.max(), pred_original.max(), pred_pca.max())
plt.plot([min_value, max_value], [min_value, max_value], color='red', linestyle='--')
plt.xlabel('Фактические расходы')
plt.ylabel('Предсказанные расходы')
plt.title('Факт и прогноз для линейной регрессии')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '07_pred_vs_actual.png', dpi=180, bbox_inches='tight')
plt.show()

## 9. Вывод

На исходных признаках линейные модели хорошо объясняют медицинские расходы, потому что в данных есть сильный фактор `smoker_yes`. Курение резко повышает прогнозируемые расходы, также положительный вклад дают возраст и BMI.

VIF не показывает критической мультиколлинеарности в исходных признаках, но PCA по заданию переводит данные в ортогональные компоненты. После PCA значения VIF компонент равны 1, однако качество прогноза не становится заметно лучше: PCA сохраняет дисперсию признаков, а не связь с целевой переменной.

Итоговое сравнение смотрим в таблице `metrics`: для этого датасета исходные признаки остаются более удобными для интерпретации, а PCA полезен как способ устранения линейной зависимости и снижения размерности.